Imports


In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime
import os




DWH = sqlite3.connect("../../Database/DWH 3.db")
SDM = sqlite3.connect("../../Database/SDM2.db")


today = datetime.now().strftime("%Y-%m-%d")


LOG_FILE = "DWH_Logboek.xlsx"



def load(table):
    try:
        return pd.read_sql_query(f"SELECT * FROM {table}", SDM)
    except:
        return pd.DataFrame()
    
def parse_time(t):
    return datetime.strptime(str(t).split(".")[0], "%H:%M:%S")

In [ ]:
def log_etl(process, action, status, count=0, error=""):
    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "process": process,
        "action": action,
        "status": status,
        "count": count,
        "error": error
    }

    df_new = pd.DataFrame([row])

    if os.path.exists(LOG_FILE):
        df_old = pd.read_excel(LOG_FILE)
        df_all = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_all = df_new

    df_all.to_excel(LOG_FILE, index=False)


In [2]:
def get_sk(table, key, value):
    mapping = {
        "Dim_Klant": "klant_sk",
        "Dim_Fiets": "fiets_sk",
        "Dim_Monteur": "monteur_sk",
        "Dim_Datum": "datum_sk",
        "Dim_Accessoire": "accessoire_sk"
    }

    sk_col = mapping[table]

    query = f"""
        SELECT {sk_col}
        FROM {table}
        WHERE {key}=?
    """

    if table in ["Dim_Klant", "Dim_Fiets"]:
        query += " AND is_actueel=1"

    df = pd.read_sql_query(query, DWH, params=(value,))
    return None if df.empty else int(df.iloc[0, 0])

In [3]:
df_klant = pd.concat([
    load("Accessoireverkoop_Klant"),
    load("Fietsverkoop_Klant")
], ignore_index=True).drop_duplicates("klantnr")

for _, row in df_klant.iterrows():

    existing = pd.read_sql_query("""
        SELECT * FROM Dim_Klant
        WHERE klantnr=? AND is_actueel=1
    """, DWH, params=(row["klantnr"],))

    geboortejaar = int(str(row["geboortedatum"])[:4])
    leeftijd = 2026 - geboortejaar

    groep = (
        "0-17" if leeftijd < 18 else
        "18-29" if leeftijd < 30 else
        "30-49" if leeftijd < 50 else "50+"
    )

    if existing.empty:
        DWH.execute("""
            INSERT INTO Dim_Klant (
                klantnr, naam, woonplaats, geslacht, geboortedatum,
                leeftijdsgroep, geldig_vanaf, geldig_tot, is_actueel
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, NULL, 1)
        """, (
            row["klantnr"], row["naam"], row["woonplaats"],
            row["geslacht"], row["geboortedatum"],
            groep, today
        ))

DWH.commit()


In [4]:
df_monteur = pd.concat([
    load("Accessoireverkoop_Monteur"),
    load("Fietsverkoop_Monteur"),
    load("Onderhoud_Monteur")
], ignore_index=True).drop_duplicates("monteurnr")

for _, row in df_monteur.iterrows():

    exists = pd.read_sql_query("""
        SELECT monteur_sk FROM Dim_Monteur WHERE monteurnr=?
    """, DWH, params=(row["monteurnr"],))

    if exists.empty:
        DWH.execute("""
            INSERT INTO Dim_Monteur (
                monteurnr, naam, uurloon,
                filiaal_naam, filiaal_provincie
            )
            VALUES (?, ?, ?, ?, ?)
        """, (
            row["monteurnr"], row["naam"], row["uurloon"],
            "UNKNOWN", "UNKNOWN"
        ))
    else:
        DWH.execute("""
            UPDATE Dim_Monteur
            SET naam=?, uurloon=?
            WHERE monteurnr=?
        """, (
            row["naam"], row["uurloon"], row["monteurnr"]
        ))

DWH.commit()

In [5]:
df_fiets = pd.concat([
    load("Fietsverkoop_Fiets"),
    load("Onderhoud_Fiets"),
    load("Fiets_Inkoop_Fiets")
], ignore_index=True).drop_duplicates("fietsnr")

for _, row in df_fiets.iterrows():

    prijsklasse = (
        "budget" if row["standaardprijs"] < 500 else
        "mid" if row["standaardprijs"] < 1500 else "premium"
    )

    existing = pd.read_sql_query("""
        SELECT * FROM Dim_Fiets
        WHERE fietsnr=? AND is_actueel=1
    """, DWH, params=(row["fietsnr"],))

    if existing.empty:
        DWH.execute("""
            INSERT INTO Dim_Fiets (
                fietsnr, soort, merk, type, kleur,
                standaardprijs, inkoopprijs, prijsklasse,
                fabrikant_naam, fabrikant_plaats,
                geldig_vanaf, geldig_tot, is_actueel
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL, 1)
        """, (
            row["fietsnr"], row["soort"], row["merk"], row["type"],
            row["kleur"], row["standaardprijs"], row["inkoopprijs"],
            prijsklasse, "UNKNOWN", "UNKNOWN", today
        ))

DWH.commit()

In [6]:
df_acc = pd.concat([
    load("Accessoireverkoop_Accessoire"),
    load("Accessoire_Inkoop_Accessoire")
], ignore_index=True).drop_duplicates("accessoirenr")

for _, row in df_acc.iterrows():

    exists = pd.read_sql_query("""
        SELECT accessoire_sk FROM Dim_Accessoire WHERE accessoirenr=?
    """, DWH, params=(row["accessoirenr"],))

    if exists.empty:
        DWH.execute("""
            INSERT INTO Dim_Accessoire (
                accessoirenr, naam, soort,
                standaardprijs, inkoopprijs,
                leverancier_naam, leverancier_woonplaats
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            row["accessoirenr"], row["naam"], row["soort"],
            row["standaardprijs"], row["inkoopprijs"],
            "UNKNOWN", "UNKNOWN"
        ))

DWH.commit()


In [7]:
df_dates = pd.concat([
    load("Fietsverkoop_Fiets_Verkoop")[["datum"]],
    load("Accessoireverkoop_Accessoire_Verkoop")[["datum"]],
    load("Onderhoud_Onderhoud")[["datum"]]
], ignore_index=True).drop_duplicates()

for d in df_dates["datum"]:
    exists = pd.read_sql_query(
        "SELECT 1 FROM Dim_Datum WHERE datum=?",
        DWH,
        params=(d,)
    )

    if exists.empty:
        dt = datetime.strptime(d, "%Y-%m-%d")
        kwartaal = (dt.month - 1)//3 + 1

        DWH.execute("""
            INSERT INTO Dim_Datum (datum, dag, maand, jaar, kwartaal)
            VALUES (?, ?, ?, ?, ?)
        """, (d, dt.day, dt.month, dt.year, kwartaal))

DWH.commit()


In [12]:
df = load("Accessoireverkoop_Accessoire_Verkoop")

inserted = 0

for _, r in df.iterrows():

    klant_sk = get_sk("Dim_Klant", "klantnr", r["klant"])
    accessoire_sk = get_sk("Dim_Accessoire", "accessoirenr", r["accessoire"])
    monteur_sk = get_sk("Dim_Monteur", "monteurnr", r["monteur"])
    datum_sk = get_sk("Dim_Datum", "datum", r["datum"])

    if None in (klant_sk, accessoire_sk, monteur_sk, datum_sk):
        continue

    totale = r["aantal"] * r["verkoopprijs"]

    DWH.execute("""
        INSERT INTO Fact_Accessoire_Verkoop (
            accessoire_verkoop,
            datum_sk,
            klant_sk,
            accessoire_sk,
            monteur_sk,
            aantal,
            verkoopprijs,
            totale_prijs
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        r["accessoire_verkoopnr"],
        datum_sk,
        klant_sk,
        accessoire_sk,
        monteur_sk,
        r["aantal"],
        r["verkoopprijs"],
        totale
    ))

    inserted += 1

DWH.commit()




In [ ]:
df = load("Onderhoud_Onderhoud")

for _, r in df.iterrows():
    datum_sk = get_sk("Dim_Datum", "datum", r["datum"])
    fiets_sk = get_sk("Dim_Fiets", "fietsnr", r["fiets"])
    monteur_sk = get_sk("Dim_Monteur", "monteurnr", r["monteur"])

    if None in (datum_sk, fiets_sk, monteur_sk):
        continue

    start = parse_time(r["starttijd"])
    end = parse_time(r["eindtijd"])

    duur = int((end - start).total_seconds() / 60)

    uurloon = pd.read_sql_query(
        "SELECT uurloon FROM Dim_Monteur WHERE monteur_sk=?",
        DWH,
        params=(monteur_sk,)
    ).iloc[0, 0]

    kosten = (duur / 60) * uurloon

    DWH.execute("""
        INSERT INTO Fact_Onderhoud (
            onderhoud, datum_sk, fiets_sk, monteur_sk,
            duur_minuten, arbeidskosten
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        r["onderhoudnr"],
        datum_sk, fiets_sk, monteur_sk,
        duur, kosten
    ))

DWH.commit()

print("DWH FULL LOAD SUCCESS")

✅ DWH FULL LOAD SUCCESS


In [9]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    DWH
)

print(tables)


                      name
0                Dim_Klant
1          sqlite_sequence
2           Dim_Accessoire
3              Dim_Monteur
4                Dim_Datum
5                Dim_Fiets
6       Fact_Fiets_Verkoop
7           Fact_Onderhoud
8  Fact_Accessoire_Verkoop


In [10]:
print(pd.read_sql_query("SELECT COUNT(*) FROM Dim_Klant", DWH))
print(pd.read_sql_query("SELECT COUNT(*) FROM Dim_Fiets", DWH))
print(pd.read_sql_query("SELECT COUNT(*) FROM Fact_Onderhoud", DWH))



   COUNT(*)
0        25
   COUNT(*)
0        75
   COUNT(*)
0       100
